# Penalty sweep demo

Refit one target across a **penalty grid** (same cells each time), then **filter factors** by stability across the path (sign flips / going to zero). `frac_zero` is an L1/elastic concept -- ridge never zeros.

## 1. Load

In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.least_squares import sweep_gene_betas

In [ ]:
# Pick a dataset (any key in CONFIGS: 'cervical' | 'breast' | 'colon').
cfg = CONFIGS['cervical']
adata = sc.read_h5ad(cfg.adata_path)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')

# Cell selection + fit settings default to the dataset's config; override inline to test.
ANNOT_COL = cfg.annot_col or cfg.cell_type_col
CELL_TYPE = cfg.cell_type or (cfg.t_cell_types[0] if cfg.t_cell_types else None)
METHOD, PENALTY, L1_RATIO, STANDARDIZE = cfg.method, cfg.penalty, cfg.l1_ratio, cfg.standardize
print('annot_col =', ANNOT_COL, '| cell_type =', CELL_TYPE,
      '| method =', METHOD, '| penalty =', PENALTY)

In [ ]:
# Optional: add cancer factors so they're available as predictors (cancers).
# library = sh.load_msigdb(cfg.dbver, cfg.categories)
# cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
# sh.add_cancer_factors(adata, {k: v for k, v in cancer_sets.items() if v}, annot_col=cfg.cell_type_col)

TARGET = cfg.single_gene or adata.uns['x_genes'][0]   # edit to any built gene (or a signature name you added)
PENALTIES = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
print('TARGET =', TARGET)

## 2. Sweep over penalties

In [ ]:
sweep = sweep_gene_betas(adata, TARGET, PENALTIES, annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                         metabolites=[], custom='all', method=METHOD, l1_ratio=L1_RATIO,
                         standardize=STANDARDIZE)
sweep.head()

## 3. Filter factors by path stability

In [ ]:
summary = sh.sweep_filter(sweep)           # per factor: min/max beta, n_sign_flips, frac_zero, stable_sign
summary['abs_max'] = summary[['min_beta', 'max_beta']].abs().max(axis=1)

# Keep factors that never flip sign AND are never zeroed across the penalty path (robust effects).
stable = summary[(summary['n_sign_flips'] == 0) & (summary['frac_zero'] == 0)]
stable.sort_values('abs_max', ascending=False).head(40)

## 4. Beta-vs-penalty paths for the top stable factors

In [ ]:
top = stable.sort_values('abs_max', ascending=False)['factor'].head(10).tolist()
fig, ax = plt.subplots(figsize=(8, 5))
for f in top:
    d = sweep[sweep['factor'] == f].sort_values('penalty')
    ax.plot(d['penalty'], d['beta'], marker='o', label=f)
ax.set_xscale('log'); ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('penalty'); ax.set_ylabel('beta'); ax.legend(fontsize=7, ncol=2)
ax.set_title(f'{TARGET}: beta paths (top stable factors)');